# 02 · Evaluate a baseline: relational retrieval on initial facet suggestion

In **initial facet suggestion** (paper §4.2), a method sees only the context q = (u, t, e) and surfaces 10 facets. It is scored by:

- **Navigation & search cost** (§3.4.1). This is the simulated effort to reach every target facet. Each target is an exact **H**it, a same-app **A**djustment (a surfaced facet of the same app serves as entry point), or a **M**iss (manual access through the app list). The cost is compared against fully manual access.
- **Retrieval metrics** (§3.4.2): Recall@10, F1@10 and Hit@10.

Relational retrieval and global popularity are fit per cross-validation fold, using the paper's exact folds. Relational retrieval needs the `[embed]` extra.

In [ ]:
from contextxr import evaluate
from contextxr.methods import RelationalRetrieval

results = evaluate(RelationalRetrieval(), "initial")
s = results.summary
print(f"manual {s['manual_cost']:.2f} -> assisted {s['suggest_cost']:.2f} ({s['cost_reduction_pct']:.1f}%)")
print(f"Recall@10 {s['recall']:.3f}  F1@10 {s['f1']:.3f}  Hit@10 {s['hit']:.3f}")

manual 21.20 -> assisted 18.39 (-13.3%)
Recall@10 0.259  F1@10 0.193  Hit@10 0.798


## One scenario's H/A/M breakdown

This is the Fig. 5 context (P04, relaxing, living room). Every record keeps the suggestions, the outcome for each target, and each cost term.

In [ ]:
record = next(r for r in results.records if r["scenario_id"] == "P04_LivingRoom_Relaxing")
print("suggestions:", record["suggestions"])
print("partition:", record["hsm"]["partition"])
for t in record["hsm"]["targets"]:
    via = f" via {t['source_facet_id']} ({t['hops']} hop)" if t["outcome"] == "A" else ""
    print(f"  {t['outcome']} {t['facet_id']:32} app_scan={t['app_scan']:.0f} facet_nav={t['facet_nav']:.0f} adjust={t['adjust']:.0f}{via}")
print({k: v for k, v in record["cost"].items() if k in ("manual", "suggest")})

Save the per-scenario log for later analysis (format: `docs/results_format.md`):

In [ ]:
import tempfile, os
path = results.to_jsonl(os.path.join(tempfile.mkdtemp(), "relational_initial.jsonl"))
print(path.name, sum(1 for _ in open(path)), "lines")